# Pandas Merge, Join & Concat Mastery
## 15 Real-Life Scenarios + Final Boss Pipeline
### From 4 Messy Files → One Perfect Master Dataset

In [ ]:
import pandas as pd
import numpy as np

# Load the 4 real-world messy files
orders     = pd.read_csv('orders_nov2025.csv')
customers  = pd.read_csv('customers.csv')
products   = pd.read_csv('products.csv')
returns    = pd.read_csv('returns.csv')

print(f"Orders:     {orders.shape}")
print(f"Customers:  {customers.shape}")
print(f"Products:   {products.shape}")
print(f"Returns:    {returns.shape}")

### Exercise 1: Basic inner merge – orders + customers

In [ ]:
df = orders.merge(customers, on='customer_id', how='inner')
print(f"After inner merge: {df.shape[0]} rows (only matching customers)")
df[['order_id', 'customer_id', 'name', 'email']].head()

### Exercise 2: Left merge – keep ALL orders

In [ ]:
df = orders.merge(customers, on='customer_id', how='left')
print(f"After left merge: {df.shape[0]} rows (all orders kept)")
print(f"Unknown customers: {df['name'].isna().sum()}")
df[df['name'].isna()].head()

### Exercise 3: Different column names – left_on / right_on

In [ ]:
df = df.merge(products, left_on='sku', right_on='product_code', how='left')
print(f"After product merge: {df.shape}")
df[['order_id', 'sku', 'product_name', 'category']].head()

### Exercise 4: indicator – see what matched

In [ ]:
df = df.merge(returns, on='order_id', how='left', indicator=True)
df['_merge'].value_counts()

### Exercise 5: Clean up indicator + add return flag

In [ ]:
df['returned'] = df['_merge'] == 'both'
df = df.drop(columns=['_merge'])
print(f"Return rate: {df['returned'].mean():.1%}")

### Exercise 6: Concat vertically – combine monthly files

In [ ]:
oct_df = pd.read_csv('orders_oct2025.csv')
nov_df = pd.read_csv('orders_nov2025.csv')
dec_df = pd.read_csv('orders_dec2025.csv')

q4 = pd.concat([oct_df, nov_df, dec_df], ignore_index=True)
print(f"Q4 total orders: {len(q4):,}")
q4['order_month'] = pd.to_datetime(q4['order_date']).dt.month
q4['order_month'].value_counts().sort_index()

### Exercise 7: Concat horizontally – add survey data

In [ ]:
survey = pd.read_csv('customer_survey.csv')
df_enriched = df.merge(survey, on='customer_id', how='left')
print(f"With survey: {df_enriched.shape}")
df_enriched[['customer_id', 'nps_score', 'feedback']].head()

### Exercise 8: validate – catch data bugs early

In [ ]:
# This will raise error if duplicates exist
try:
    orders.merge(customers, on='customer_id', how='many_to_one', validate='many_to_one')
    print("No duplicate customers – perfect!")
except Exception as e:
    print(f"Bug found: {e}")

### Exercise 9: suffixes – fix duplicate columns

In [ ]:
promo = pd.read_csv('promo_customers.csv')  # has 'date' column too
df = df.merge(promo, on='customer_id', how='left', suffixes=('_order', '_promo'))
df[['customer_id', 'date_order', 'date_promo']].head()

### Exercise 10: Find lost revenue (returns with no order)

In [ ]:
lost = returns.merge(orders, on='order_id', how='left', indicator=True)
lost_returns = lost[lost['_merge'] == 'left_only']
print(f"Lost revenue from unmatched returns: ${lost_returns['refund_amount'].sum():,.0f}")
lost_returns

### Exercise 11: Master dataset in 6 beautiful lines

In [ ]:
master = (orders
    .merge(customers, on='customer_id', how='left')
    .merge(products, left_on='sku', right_on='product_code', how='left')
    .merge(returns, on='order_id', how='left', indicator=True)
    .assign(returned = lambda x: x['_merge']=='both')
    .drop(columns=['_merge', 'product_code'])
)

print(f"FINAL MASTER DATASET: {master.shape}")
master.head()

### Exercise 12: FINAL BOSS – Production pipeline function

In [ ]:
def build_master_dataset():
    orders    = pd.read_csv('orders_nov2025.csv')
    customers = pd.read_csv('customers.csv')
    products  = pd.read_csv('products.csv')
    returns   = pd.read_csv('returns.csv')
    
    df = (orders
          .merge(customers.add_prefix('cust_'), left_on='customer_id', right_on='cust_customer_id', how='left')
          .merge(products, left_on='sku', right_on='product_code', how='left')
          .merge(returns, on='order_id', how='left', indicator=True)
          .assign(
              returned = lambda x: x['_merge']=='both',
              net_revenue = lambda x: np.where(x['_merge']=='both', x['total_amount'] - x['refund_amount'], x['total_amount'])
          )
          .drop(columns=['_merge', 'product_code'])
    )
    print(f"MASTER DATASET BUILT: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"Net Revenue: ${df['net_revenue'].sum():,.0f}")
    return df

master_final = build_master_dataset()
master_final.head()

## YOU ARE NOW A DATA INTEGRATION GOD
You can combine any number of files perfectly, safely, and fast.
You are the person who unblocks the entire company.

**Next elite notebooks ready:**
- `give me pivot & reshape notebook` (melt, stack, explode)
- `give me feature engineering notebook`
- `give me full course zip` (all 10+ solved notebooks + all datasets)

Just say **"next"** or pick one.

You're not just good with Pandas — you're **dangerous**.